In [1]:
from euriai.langchain import create_chat_model

chat_model = create_chat_model(
    api_key="your_euri_api_token_here",
    model="gpt-4.1-nano",
    temperature=0.7
)

response = chat_model.invoke("What is artificial intelligence?")
print(response.content)

Artificial intelligence (AI) refers to the simulation of human intelligence processes by machines, particularly computer systems. These processes include learning (the ability to improve performance based on experience), reasoning (the ability to draw conclusions from data), problem-solving, understanding language, perception (such as vision or speech recognition), and decision-making. AI can be categorized into narrow AI, which is designed for specific tasks (like voice assistants or recommendation systems), and general AI, which would have the ability to perform any intellectual task a human can do. The development of AI aims to create systems that can operate autonomously, adapt to new information, and perform complex tasks that typically require human intelligence.


In [2]:
import pydantic
print(pydantic.__version__)

2.12.5


In [3]:
import ast,re,math,os,sys
os.environ["euri-api-key"] ="your_euri_api_token_here"
from euriai.langchain import create_chat_model
from langchain.agents import create_react_agent,AgentExecutor
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate,MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableLambda,RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory ,InMemoryChatMessageHistory as ChatMessageHistory
from langchain_core.tools import BaseTool,tool
from langchain.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory,
    CombinedMemory,
    ReadOnlySharedMemory,
    ConversationEntityMemory
)

ImportError: cannot import name 'create_react_agent' from partially initialized module 'langchain.agents' (most likely due to a circular import) (c:\conda\envs\agentai\Lib\site-packages\langchain\agents\__init__.py)

In [4]:
from euriai.langchain import create_chat_model

llm_default = create_chat_model(
    api_key="your_euri_api_token_here",
    model="gpt-4.1-nano",
    temperature=0.7
)


In [5]:
@tool("calculator",return_direct=True)
def calculator(expression: str) -> str:
    """Evaluate a numeric math expression . supprts + , -,*,/,**,parentheses and all kind of mathamatical functions"""
    allowed_nodes =(
        ast.Expression,
        ast.UnaryOp,
        ast.unaryop,
        ast.BinOp,
        ast.operator,
        ast.Num,
        ast.Load,
        ast.pow,
        ast.FunctionDef,
        ast.Module,
        ast.Expr,
        ast.Call,
        ast.Name,
        ast.arguments,
        ast.args,
        ast.Constant
    )
    
    allowed_names = {k:v for k,v in vars(math).items() if not k.startswith("_")}
    allowed_names.update({"abs": abs, "round": round,"min": min,"max": max})
    node = ast.parse(expression, mode="eval")
    
    
    for n in ast.walk(node):
        if not isinstance(n, allowed_nodes):
            raise ValueError(f"Expression contains invalid node {type(n)}")
        if isinstance(n, ast.Name) and n.id not in allowed_names:
            raise ValueError(f"Expression contains invalid name {n.id}")
    code = compile(node, "<string>", "eval")
    return str(eval(code, {"__builtins__": {}}, allowed_names))

NameError: name 'tool' is not defined

In [ ]:
tools_math = [calculator]

In [ ]:
react_prompt_math = ChatPromptTemplate.from_messages([(
    "system","you are a precise math assistant , you can use thse tools:\n{tools}\n"
    "when you are going to use this tools follw these instruction exactly the same format:\n"
    "Questions :......\nthought ...\nAction by using one of the tools access that you have [{tool_names}]\n "
    "alwasy finish with final give me a numeric answer to the question"),
    ("human","questions: {input}\n{agent_scratchpad}")
])

In [ ]:
agent_math = create_react_agent(
    llm=llm_default,
    prompt=react_prompt_math,
    tools=tools_math
)

In [ ]:
memory_math = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    input_key="input",
    output_key="output",
)

In [ ]:
math_exector = AgentExecutor(
    agent=agent_math,
    tools=tools_math,
    memory=memory_math,
    verbose=True,
    handle_parsing_errors=(
        "you did not follow the instruction that i have given you to use the tools"
    ),
)

In [ ]:
@tool("kb_search", return_direct=True)
def kb_search(query: str) -> str:
    """a mock function to search from a knowledge base"""
    knowledge_base = {
        "what is the capital of france": "The capital of France is Paris.",
        "who is the president of the united states": "The president of the United States is Joe Biden.",
        "what is the largest mammal": "The largest mammal is the blue whale.",
    }
    query = query.lower().strip("?")
    for key,answer in knowledge_base.items():
        if key in query:
            return answer
    return "I don't know the answer to that question."

In [ ]:
tools_kb = [kb_search]

In [ ]:
react_prompt_kb = ChatPromptTemplate.from_messages([
    ("system","you are a helpful assistant that can answer question based on your knowledge base and you can use the following tools:\n{tools}\n"
    "when you are going to use this tools follw these instruction exactly the same format:\n"
    "Questions :......\nthought ...\nAction by using one of the tools access that you have [{tool_names}]\n "
    "alwasy finish with final answer to the question that you are able to search from my knowledge base"),
    ("human","questions: {input}\n{agent_scratchpad}")
])

In [ ]:
agent_kb = create_react_agent(
    llm=llm_default,
    prompt=react_prompt_kb,
    tools=tools_kb
)

In [ ]:
mem_kb = ConversationBufferWindowMemory(
    memory_key="chat_history",
    return_messages=True,
    input_key="input",
    output_key="output",
    k=5
)

In [ ]:
kb_executor = AgentExecutor(
    agent=agent_kb,
    tools=tools_kb,
    memory=mem_kb,
    verbose=True,
    handle_parsing_errors=(
        "you did not follow the instruction that i have given you to use the kb  tools"
    ),
)

In [ ]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a router. Read the user message and output exactly one token:\n"
     "- MATH: if it asks for calculations.\n"
     "- KB: otherwise (tech education, RAG, agents, general know-how).\n"
     "Output only MATH or KB."),
    ("human", "{input}")
])

In [ ]:
router_chain = router_prompt|llm_default|StrOutputParser()

In [ ]:
def _dispatcher(inputs:dict):
    """it will take new inputs from human and it will also attach hitory from memory """
    user_msg = inputs["input"]
    choice = router_chain.invoke({"input": user_msg}).strip().upper()
    print(f"router choice : {choice}")
    if choice == "MATH" or choice == "Math" or choice == "math":
        return math_exector.invoke({"input": user_msg})
    elif choice == "KB" or choice == "Kb" or choice == "kb":
        return kb_executor.invoke({"input": user_msg})
    else:
        return "I can only answer math and kb related questions"

In [ ]:
dispatcher = RunnableLambda(_dispatcher)

In [ ]:
_sessions = {}

In [ ]:
def _get_history(session_id:str):
    if session_id not in _sessions:
        _sessions[session_id] = ChatMessageHistory()
    return _sessions[session_id]

In [ ]:
orchestrator = RunnableWithMessageHistory(
    runnable=dispatcher,
    get_session_history=_get_history,
    input_key="input",
    history_messages_key="history"
)

In [ ]:
cfg = {"configurable":{"session_id":"user1"}}

In [ ]:
print(orchestrator.invoke({"input":"what is the capital of france?"},config=cfg))

In [ ]:
print(orchestrator.invoke({"input":"what is sin(90)"},config=cfg))

In [ ]:
1.whatsapp tool
2.email tool
3.sms tool
4.telegram tool
5.calander tool